## **Transformers Walkthrough**
This script is a walk-through of the Transformers architecture. We demonstrate the various Transformers component in a language translation task.

## Load pre-trained model and tokenizer

In [2]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

checkpoint = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(checkpoint,
                                          additional_special_tokens=['<2ceb>','<2es>','<2en>','<2hil>','<2cbk>']
                                          )
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")
model = model.to(device)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

...

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
model.config

MT5Config {
  "architectures": [
    "MT5ForConditionalGeneration"
  ],
  "bos_token_id": null,
  "classifier_dropout": 0.0,
  "d_ff": 1024,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "mt5",
  "num_decoder_layers": 8,
  "num_heads": 6,
  "num_layers": 8,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": true,
  "tokenizer_class": "T5Tokenizer",
  "transformers_version": "5.16.1",
  "use_cache": true,
  "vocab_size": 250112
}

In [5]:
'''
ANSWER.
Explore the mt5 vocabulary using get_vocab().
'''

vocab = tokenizer.get_vocab()

In [6]:
'''
Question 1. What is the size of the mT5 vocabulary?
The actual vocabulary size is less than 5 since we added some special tokens (language tags)

Answer: 
The actual vocabulary size is 250,100. Less than 5 of the special tokens added.
'''

len(vocab)

250105

In [7]:
'''
Question 2. What is the structure of the mT5 vocabulary?

Answer: 
The vocabulary is dictionary of tokens mapped with its ID number. Tokens are subwords from different languages 
while "_" at the front indicates a subword as a start word.
'''
vocab

{'▁இப்பட': 181200,
 'амино': 190086,
 'нати': 40618,
 '▁tagasi': 32979,
 '▁Canal': 43405,
 'confidenti': 194421,
 'ாவிட': 189877,
 'кыз': 170854,
 'kateri': 27367,
 '3558': 225853,
 '▁em': 724,
 'punktet': 177907,
 '▁Simula': 136246,
 'សម្រួល': 203060,
 'picas': 200809,
 '▁Oliv': 92026,
 'կցի': 222623,
 'seres': 69372,
 '2140': 149591,
 'індік': 122805,
 'MURA': 138410,
 '▁keele': 91667,
 '▁Vilnius': 39966,
 'курса': 101177,
 'ၾကသည္။': 166399,
 'EYA': 177443,
 '스가': 129930,
 'රුම': 123651,
 '▁உத்தர': 186403,
 'льному': 210254,
 'anıyorum': 217195,
 'venster': 176930,
 '▁disponibil': 95224,
 '_16': 149224,
 'gazdag': 122459,
 'ήμερ': 186332,
 '▁تد': 21437,
 '▁expansi': 142675,
 '▁tayyor': 99213,
 'నార్': 216229,
 'hord': 168074,
 'ꜗ': 244746,
 '▁കടന്നു': 178243,
 '仔': 110483,
 '▁minori': 133166,
 'wh': 30023,
 'ράγ': 201562,
 '▁xxx': 5965,
 'livs': 153313,
 'jetër': 121876,
 'اعب': 84541,
 '▁kongre': 171995,
 '🏅': 243874,
 'տիվ': 105959,
 'Sida': 149648,
 'alueet': 213028,
 'rairie': 12

In [8]:
'''
Question 3. What is the token ID for the special token '<pad>'?
Remember that this value may differ in other models.

Answer:
Its ID is 0
'''
vocab['<pad>']

0

## Explore Input Embedding
This section is not really a part of model training but I just wanted to show you how Transformers handle token embeddings.

In [9]:
input_text = "The black cat sat on the couch and the brown dog slept on the rug."
inputs = tokenizer(input_text, return_tensors="pt")
input_ids = inputs["input_ids"]
input_tokens = tokenizer.convert_ids_to_tokens(input_ids.flatten())
print(input_ids)
print(input_tokens)

tensor([[  486,  5866, 10292, 31533,   351,   287,   259, 78645,   305,   287,
         44442, 10990, 82212,   270,   351,   287, 23573,   260,     1]])
['▁The', '▁black', '▁cat', '▁sat', '▁on', '▁the', '▁', 'couch', '▁and', '▁the', '▁brown', '▁dog', '▁slep', 't', '▁on', '▁the', '▁rug', '.', '</s>']


In [10]:
# Extract the input embeddings directly
embedding_layer = model.shared  # nn.Embedding; embedding matrix shared between the encoder and decoder
embedding_layer.to('cpu') # I have to force cpu computing because pytorch have issues with m3
input_embeddings = embedding_layer(input_ids).to('cpu')  # shape: [batch_size, seq_len, hidden_size]

print(input_embeddings.shape) 

torch.Size([1, 19, 512])


In [11]:
'''
Cosine similarity measures the cosine of the angle between two vectors in a high-dimensional space.
+1 → Vectors point in the same direction → Maximum similarity
 0 → Vectors are orthogonal (unrelated) → No similarity
−1 → Vectors point in opposite directions → Maximum dissimilarity
'''
cosine_similarity = torch.nn.CosineSimilarity(dim=0, eps=1e-8)
print(f'similarity between {input_tokens[1]} and {input_tokens[2]}: {cosine_similarity(input_embeddings[0][1],input_embeddings[0][2])}')
print(f'similarity between {input_tokens[1]} and {input_tokens[10]}: {cosine_similarity(input_embeddings[0][1],input_embeddings[0][10])}')
print(f'similarity between {input_tokens[2]} and {input_tokens[11]}: {cosine_similarity(input_embeddings[0][2],input_embeddings[0][11])}')

similarity between ▁black and ▁cat: 0.11180498450994492
similarity between ▁black and ▁brown: 0.3889656364917755
similarity between ▁cat and ▁dog: 0.23778599500656128


## Tokenization

In [16]:
from datasets import load_from_disk
dataset_dir = '/kaggle/input/datasets/normanjazul/small-dataset/small_dataset'
dataset = load_from_disk(dataset_dir)

In [19]:
'''
Question 4. Describe the dataset in terms of how the samples are structured.

Answer:
The dataset is a list of dictionaries with 3 members: train, test, and valid. 
Each members are also a list of features with 200 members each. 
Each rows here are source sentence with target sentence translation and the language for each sentece.
'''
dataset

DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'language_pair'],
        num_rows: 200
    })
    test: Dataset({
        features: ['source', 'target', 'language_pair'],
        num_rows: 200
    })
    valid: Dataset({
        features: ['source', 'target', 'language_pair'],
        num_rows: 200
    })
})

In [20]:
'''
Prepare inputs and targets as lists.
'''
def preprocess_function(examples):
    inputs = [str(example) for example in examples['source']]
    targets = [str(example) for example in examples['target']]
    return tokenizer(inputs, text_target=targets, max_length=128 ,truncation=True, padding='max_length', return_tensors="pt")


'''
Map() applies a function to each example or batch in your dataset to transform or tokenize it, typically before training.
This function can create new rows and columns or add prefixes.
'''
# Tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Ran in kaggle notebook so i need to redirect the output to the working dir
tokenized_dataset = dataset.map(
    preprocess_function, 
    batched=True,
    cache_file_names={
        "train": "/kaggle/working/train_tokenized.arrow",
        "test":  "/kaggle/working/test_tokenized.arrow",
        "valid": "/kaggle/working/valid_tokenized.arrow",
    },
)

In [21]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'language_pair', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['source', 'target', 'language_pair', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    valid: Dataset({
        features: ['source', 'target', 'language_pair', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})

In [22]:
#load tokenized dataset
from datasets import load_from_disk
tokenized_dir = '/kaggle/input/datasets/normanjazul/m2n-tokenized-small/m2n_tokenized_small'
tokenized_dataset = load_from_disk(tokenized_dir)

In [24]:
'''
Explore the tokenized dataset.
Question 5: Describe the tokenized dataset in terms of how the samples are structured.

Answer: 
This dataset has similar structure as of the previous one. The only difference is that the 3 members 
now has 1000 rows each. Each row are also given an attention mask and a label. The language pair is removed.
'''
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['source', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
    valid: Dataset({
        features: ['source', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})

In [25]:
'''
Retrieve the first input sample of the tokenized training dataset and convert this back to tokens using
tokenizer.convert_ids_to_tokens()
Question 6: Describe the tokens in the tokenized dataset.

Answer:
The first token is a label of the language of the sentence like <2ceb> (To Cebuano).
The following tokens are regualar subwords with "_" and some with don't (not a start word). 
The sentence ends with a stop token </s> and a bunch of padding <pad> afterwards.
'''
print(tokenizer.convert_ids_to_tokens(tokenized_dataset['train']['input_ids'][0]))
print(dataset['train']['source'][0])


['<2ceb>', '▁Sa', '▁piha', 'k', '▁nga', '▁', 'bahin', ',', '▁ang', '▁mahi', 'g', 'ugma', 'on', '▁kag', '▁', 'balanse', '▁nga', '▁', 'disiplin', 'a', '▁mag', 'ahana', 's', '▁sa', '▁hun', 'ahuna', '▁sang', '▁bata', '▁kag', '▁mag', 'adi', 'hon', '▁sang', '▁', 'iya', '▁b', 'atasan', '.', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad

## Finetuning
Since we only have few training data, it is better (in practice) to fine-tune a pre-trained model. 
If you decide to train from scratch using the configuration of a model, you may do so by creating an instance of the model without loading the weights. Here's an example:

    # Load mt5 configuration
      vocab_size = 30000
      config = MT5Config.from_pretrained('google/mt5-base',vocab_size=vocab_size)

    # Create a model from an existing configuration, does not load pre-trained weights
      model = TFMT5ForConditionalGeneration.from_config(config)
      optimizer = AdamWeightDecay(learning_rate=0.001)

      model.compile(optimizer=optimizer)

You can also train your own tokenizer and build your own vocabulary. Please check the Huggingface documentation on how to do these.

In [26]:
# view the model's configuration
# mT5 checkpoints vary in  the model ffn size
model.config

MT5Config {
  "architectures": [
    "MT5ForConditionalGeneration"
  ],
  "bos_token_id": null,
  "classifier_dropout": 0.0,
  "d_ff": 1024,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "mt5",
  "num_decoder_layers": 8,
  "num_heads": 6,
  "num_layers": 8,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": true,
  "tokenizer_class": "T5Tokenizer",
  "transformers_version": "5.16.1",
  "use_cache": true,
  "vocab_size": 250112
}

In [27]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

# Define training arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/outputs",
    per_device_train_batch_size=8,  #Trainer handles batching!
    per_device_eval_batch_size=8,
    learning_rate=2e-5,             #The learning rate controls how big a step the optimizer takes when updating the model’s parameters (weights) during training.
    weight_decay=0.01,              #Regularization; penalizes large weights in the model by adding a term to the loss function that grows with the size of the weights.
    num_train_epochs=3,
    eval_strategy="epoch",          #1 epoch = 1 full pass through the entire training dataset
    torch_empty_cache_steps = 2,    #1 step = 1 forward pass + 1 backward pass + 1 weight update; if batch_size =  64, then 1 step = processing 64 training examples
    save_strategy = 'epoch',        #for saving checkpoints
    logging_strategy="steps",       #to log training loss
    logging_steps=10
)

#Data collator is used at runtime by the DataLoader to collate and pad a batch of already-tokenized examples
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, return_tensors="pt", label_pad_token_id = 0, padding=True)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['valid'],
    # Include data_collator to format the data correctly for MT5
    data_collator= data_collator 
)

#Start training
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,91.483099,85.662415
2,83.750134,77.241142
3,84.137585,74.753532


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=88.37743430162864, metrics={'train_runtime': 259.4487, 'train_samples_per_second': 11.563, 'train_steps_per_second': 0.728, 'total_flos': 384169697280000.0, 'train_loss': 88.37743430162864, 'epoch': 3.0})

## Evaluating Model using BLEU

In [28]:
tokenized_dataset['test']

Dataset({
    features: ['source', 'target', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})

In [33]:
import evaluate
from tqdm import tqdm


# Set device (MPS for Mac, CUDA for NVIDIA, CPU fallback)
#device = torch.device("mps" if torch.backends.mps.is_available()
#                      else "cuda" if torch.cuda.is_available()
#                      else "cpu")
#model = model.to(device)
model.eval() # use this when evaluating the model

# Load BLEU metric
bleu = evaluate.load("sacrebleu")

# Define test inputs and references
# Example structure:
# tokenized_dataset['test'] = {'input_ids': [...], 'attention_mask': [...], 'labels_text': [...]}

batch_size = 16  # adjust depending on available memory
all_predictions = []
all_references = []

# Split test set into batches
n = len(tokenized_dataset['test']['input_ids'])
for i in tqdm(range(0, n, batch_size)):
    input_ids = torch.tensor(tokenized_dataset['test']['input_ids'][i:i+batch_size]).to(device)
    attention_mask = torch.tensor(tokenized_dataset['test']['attention_mask'][i:i+batch_size]).to(device)

    with torch.no_grad(): #disable gradient calculation since we are evaluating the model
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=64,
            num_beams=4
        )

    decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_predictions.extend(decoded_preds)

    batch_refs = tokenized_dataset['test']['target'][i:i+batch_size]
    all_references.extend([[ref] for ref in batch_refs])  # BLEU expects list of lists

# Compute BLEU score
bleu_result = bleu.compute(predictions=all_predictions, references=all_references)
print(f"\nBLEU Score: {bleu_result['score']:.2f}")


100%|██████████| 63/63 [00:35<00:00,  1.77it/s]



BLEU Score: 0.04


In [34]:
tokenized_dataset['test']['input_ids'][0]

[250100,
 259,
 219666,
 34206,
 1183,
 2182,
 200988,
 38466,
 117468,
 7217,
 200988,
 4697,
 1487,
 327,
 4750,
 291,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [36]:
# Generate translations
inputs = [str(s) for s in tokenized_dataset['test']['source']]
references = [str(t) for t in tokenized_dataset['test']['target']]

model_inputs = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=model_inputs['input_ids'],
        attention_mask=model_inputs['attention_mask'],
        max_length=64
    )

preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
# Compute BLEU score
bleu = evaluate.load("sacrebleu")
bleu_result = bleu.compute(predictions=preds, references=[[ref] for ref in references])

print(f"\nBLEU Score: {bleu_result['score']:.2f}")


BLEU Score: 0.04


In [37]:
# Fix nbconvert "KeyError: 'state'" during [Save & Run All]
import re, pathlib
import nbconvert.filters.widgetsdatatypefilter as wdf

f = pathlib.Path(wdf.__file__)
src = f.read_text()
fixed = re.sub(
    r"metadata\[['\"]widgets['\"]\]\[WIDGET_STATE_MIMETYPE\]\[['\"]state['\"]\]",
    'metadata.get("widgets", {}).get(WIDGET_STATE_MIMETYPE, {}).get("state", {})',
    src,
)
if fixed != src:
    f.write_text(fixed)
    print("nbconvert widget filter patched")
else:
    print("nothing to patch (already fixed or version changed)")

/usr/local/lib/python3.13/dist-packages/mistune.py:435: SyntaxWarning: invalid escape sequence '\|'
  cells[i][c] = re.sub('\\\\\|', '|', cell)
/usr/local/lib/python3.13/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'
  text = re.sub(r'_', '\_', text) # Escape underscores in display text


nbconvert widget filter patched


## AI Use Declaration

- I declare upon my honor that I only used AI to understand the code and the concepts in this exercise.